In [ ]:
# @title Install necessary libraries
!pip install google-genai google-cloud-storage pypdf google-colab
print("Libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.6 MB/s eta 0:00:00
Libraries installed.


In [ ]:
# @title Import libraries and authenticate
from google import genai
from google.cloud import storage
from google.colab import auth
from google.genai.types import Part, GenerateContentConfig # Import Part for file input
import pypdf
import os
import json
import textwrap # For printing long text nicely
from IPython.display import display, Markdown # For better output formatting

# Authenticate the Colab user for GCP access (like GCS)
print("Authenticating user for Google Cloud access...")
auth.authenticate_user()
print("Authentication successful.")

Authenticating user for Google Cloud access...
Authentication successful.


In [ ]:
# @title Configure Google Cloud Settings and connect Google Cloud Storage
#    Replace with your actual Project ID, Bucket Name, and Prefix (folder)
GCP_PROJECT_ID = "extraction-papers-talhouk2"
GCS_BUCKET_NAME = "tlab-extractionbucket3"
GCS_PREFIX = "papers3" # e.g., "papers/neuroscience/" or "" for root
LOCATION = "us-central1" # location you want to use for data procssing

# Optional: Set project for GCS client (often inferred after auth, but explicit is safer)
try:
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    print(f"Google Cloud Storage client initialized for project '{GCP_PROJECT_ID}'.")
except Exception as e:
     print(f"Error initializing GCS Client: {e}. Trying without explicit project ID.")
     try:
         storage_client = storage.Client()
         inferred_project = storage_client.project
         print(f"Google Cloud Storage client initialized using inferred project '{inferred_project}'.")
         GCP_PROJECT_ID = inferred_project # Update if inferred
     except Exception as e2:
         print(f"FATAL ERROR: Could not initialize GCS Client: {e2}")
         storage_client = None # Ensure it's None if failed

Google Cloud Storage client initialized for project 'extraction-papers-talhouk2'.


In [ ]:
# # @title Model Selection
MODEL_NAME = "gemini-2.0-flash-001"
client = genai.Client(vertexai=True, project=GCP_PROJECT_ID, location=LOCATION)
print(f"Using Gemini model: {MODEL_NAME}")

Using Gemini model: gemini-2.0-flash-001


In [ ]:
##@title Define Entities To Extract
ENTITIES_TO_EXTRACT = {
   #"Title of Paper": "Extract the title of the paper, e.g., POLE exonuclease domain mutation predicts long progression-free survival in grade 3 endometrioid carcinoma of the endometrium",
   # "Author list": "Extract the full list of authors, e.g., DeRycke M, Andersen J.D., Harrington K.M., Pambuccian S.E., Kalloger S.E., Boylan K.L.M., et al.",
   "Histology": "Based on the primary cancer type under study, classify the paper as one of the following: Ovarian (including ovarian cancers with endometrioid histology or arising from endometriosis, such as endometriosis-associated ovarian cancer), Endometrial (only if the primary cancer is located in the endometrium), Both (if both ovarian and endometrial cancers are studied as distinct primary cancers), Neither (if neither is a focus of the study), Do not classify as 'Endometrial' just because of endometrioid histology unless the primary site is explicitly the uterus. Likewise, do not choose 'Both' unless both cancers are discussed as distinct primary diseases.”",
    #"Histology subtype":"""Extract up to 5 distinct histological cancer subtypes discussed in the study, focusing on the most prominently analyzed or referenced.

#Use the following controlled vocabulary only (do not invent new labels):

#Endometrial Cancer

#Endometrioid Endometrial Cancer

##Non-Endometrioid Endometrial Cancer

#Endometrial Serous Carcinoma

#Endometrial Clear Cell Carcinoma

#Uterine Papillary Serous Carcinoma

#Endometrial Mucinous Carcinoma

#Small Cell Carcinoma of the Endometrium

#Malignant Mixed Mesodermal Tumour (carcinosarcoma)

#Not otherwise specified Endometrial Adenocarcinoma

#Endometrioid Ovarian Carcinoma

#Clear Cell Ovarian Carcinoma

#Endometriosis-associated Ovarian Cancer

#Serous Ovarian Carcinoma

#High-Grade Serous Ovarian Carcinoma

#Important:

#Do not confuse Endometrioid Ovarian Carcinoma with Endometriosis-associated Ovarian Cancer, they are distinct subtypes.

#List only the most frequently or centrally analyzed subtypes.

#If more than 5 are mentioned, return the top 5 and add ""Other"" at the end.

#Format your answer as a comma-separated list, e.g.:
#Endometrioid Endometrial Cancer, High-Grade Serous Ovarian Carcinoma, Endometrial Serous Carcinoma, Clear Cell Ovarian Carcinoma, Other""

 # "Publisher":"Extract the publisher, e.g., Gynecol Oncol",
#      "N of total cohort":"Extract the full sample size of the patient cohort that was analysed in the study. Carefully evaluate if multiple cohorts are mentioned and add them up. Return only one number, e.g., 192",
#      "Date of diagnosis":"Extract the date range of when the patients in the cohort received the first diagnosis, e.g., 2005 - 2011",
    # "Primary location of study":"Extract the primary city and country of the study, e.g,. Vancouver, Canada",
    # "Treatments applied in the study": "Extract a detailed summary of all treatments applied to patients in the study. Include initial treatments (e.g., surgery, debulking, salpingo-oophorectomy), chemotherapy (specify neoadjuvant or adjuvant, and agents like platinum-based, taxol, cisplatin), radiation therapy (e.g., external beam, vaginal brachytherapy), hormonal therapy (e.g., megestrol, progesterone IUS), and any targeted or immunotherapies. For each treatment, report the number of patients and percentages if available. State clearly if patients received no treatment. If treatments differ by cancer subtype, report separately. If only general treatment statements are given (e.g., 'all received platinum-based chemotherapy after surgery'), extract those directly. If treatment information is not available, return 'N/A'. Return answers in a structured, itemized format with precise counts and treatment types.",
    # "Analysis":"Extract the technology used to perform the analysis of the primary samples, e.g., IHC TMA",
   # "Analysed Biomarkers":"Extract which biomarkers were analysed in the study, e.g., POLE, p53, MMR, L1CAM, BRCA1. Only extract the top 3 most significant biomarkers, often mentioned in the methods and/or abstract and/or conclusion",
    # "Significant in multivariate (1), univariate (2), or both (3), analysis not significant (4) or unknown (5)":"Determine whether the biomarker or treatment of interest showed statistical significance in univariate analysis, multivariate analysis, both, neither, or if unknown/not reported (codes: 1 = multivariate only, 2 = univariate only, 3 = both, 4 = neither, 5 = unknown), where univariate analysis examines one variable at a time and multivariate analysis adjusts for multiple variables (often described using terms like “Cox regression,” “adjusted for,” or “independent predictor”); if statistical significance is reported without mention of adjustment or multivariate methods, assume univariate analysis.",
    # "Did the identified biomarker or treatment improve prognosis? Prognosis: Improved (1), Worse (2), Both (3), not significant (4), unknown (5)":"Determine the overall effect of the identified biomarker(s) or treatment on patient prognosis and return only one code—1 = improved, 2 = worse, 3 = both (if different subgroups or biomarkers show opposing effects), 4 = no significant effect, 5 = unknown—and if there are mixed effects, return 3 with a brief rationale.",
    # "N histotype":"Extract and explicitly state the histotype composition of the full study cohort. Only report histotypes, such as: Endometrial Carcinoma, Endometrioid Endometrial Carcinoma, Endometrioid Ovarian Carcinoma, Clear Cell Ovarian Carcinoma, Serous Ovarian Carcinoma, etc. Provide exact numbers for each histotype if available. If only percentages are given, calculate and report estimated numbers based on total cohort size. Do not include molecular subtypes, treatment information, or any other clinical characteristics. If no histotype breakdown is provided, return 'N/A'. The output should be limited strictly to histotype cohort composition.",
    # "Was a recent histological review (using 2014 WHO criteria) performed?":"The 2014 WHO criteria specify that the study should provide detailed diagnostic criteria, including histological features, grading and immunohistochemical markers; Or whether there were one or more pathologists who reviewed cases. Determine if this information is available and respond with Yes or No",
    # "5-Year Survival rates (overall survival unless states otherwise)":"Extract the 5-year survival rate, prioritizing overall survival (OS). If OS is not explicitly stated in the main text, check tables, figures, or Kaplan-Meier plots. If only a different survival type is available (e.g., progression-free survival [PFS] or disease-free survival [DFS]), report it and clearly specify the type. Format the result as: 'OVERALL SURVIVAL: 88.3%' or 'PFS: 76.1%'. If no survival data is found anywhere in the paper, return: 'NOT REPORTED'.",
    # "Mean age of participants":"Extract the mean age of participants. If mean is not available, provide the median age. Always specify which is used. Format: MEAN: 42.9, MEDIAN: 45.3. If neither is found, return: AGE NOT REPORTED",
     "% of participants with clinicopathological features (Stage I or II)":"Extract the percentage of study participants with disease Stage I or Stage II tumors, e.g., 97%",
    # "REMARK Critera":"Check if the study explicitly states that it followed REMARK criteria. Look for phrases such as:'according to the REMARK criteria','REMARK reporting criteria','in accordance with REMARK guidelines','REMARK recommendations', or any mention of 'REMARK' in reference to study design or analysis. If such a phrase is found, return: 'Yes', If there is no mention of REMARK criteria or guidelines, return: 'No', Do not infer based on formatting or study structure — rely only on explicit textual mention.",
    # "Conclusion of the study":"Extract a brief summary of the conclusion of the study, e.g., ProMisE subgroups are associated with patient outcome in both univariate and multivariate analysis."

}

In [ ]:
# @title Helper Functions
import re

def list_pdfs_in_gcs(client, bucket_name, prefix):
    """Lists PDF files in a GCS bucket/prefix."""
    if not client:
        print("GCS client not initialized. Cannot list files.")
        return []
    print(f"Listing PDFs in gs://{bucket_name}/{prefix}...")
    try:
        bucket = client.get_bucket(bucket_name)
        blobs = client.list_blobs(bucket, prefix=prefix)
        pdf_files = [f"gs://{bucket_name}/{blob.name}" for blob in blobs if blob.name.lower().endswith(".pdf") and blob.size > 0]
        print(f"Found {len(pdf_files)} PDF files.")
        return pdf_files
    except Exception as e:
        print(f"Error listing GCS files: {e}")
        return []

def extract_entities_with_gemini_uri(gcs_uri, entities):
    """Uses Gemini to extract specified entities directly from a PDF URI."""
    if not gcs_uri or not gcs_uri.startswith("gs://"):
        return {"error": "Invalid GCS URI provided"}

    print(f"  Preparing request for Gemini with URI: {gcs_uri}")
    # Create a Part object representing the PDF file in GCS
    try:
        pdf_file = Part.from_uri(
            file_uri=gcs_uri,
            mime_type="application/pdf" # Crucial for the model to know the file type
        )
    except Exception as e:
        print(f"  Error creating Part from URI {gcs_uri}: {e}")
        return {"error": f"Failed to create Part from URI: {e}"}

    system_instruction=(
        """Analyze the provided PDF file (research paper) and extract the entities given to you in your prompt.
        Provide the output strictly in JSON format with the keys from the prompt.
        If an entity cannot be found or is not applicable, use the value "n/a".
        JSON Output:"""
    ),

    # Construct the prompt, instructing the model to analyze the *provided file*
    print(f"  Calling Gemini API for {gcs_uri}...")
    results = {}

    for entity in entities:
        print(f"  Processing entity: {entity}")
        kg_context = get_kg_context(entity, kg_df)

        # 3b. Add the KG facts to the prompt (if any)
        prompt = f"""
        You are extracting the following entity from the research paper:

        **Entity**: {entity}

        **Instructions**: {entities[entity]}

        Use the following known relationships from a biomedical knowledge graph to help you:
        {kg_context}

        Please return the result strictly in the following JSON format:
        {{
          "{entity}": {{
            "value": "The extracted information",
            "confidence": ["confidence score from 0 to 100 as a percentage", a short one sentence rationale for why the confidence score was given]
          }}
        }}

        If the entity is not found, set "value" to "n/a" and "confidence" to 0.
        """

        # Make the API call with both the prompt and the file Part
        try:
            # The request contains both the text prompt and the file reference
            response = client.models.generate_content(
                model=MODEL_NAME,
                config=GenerateContentConfig(
                    system_instruction=system_instruction),
                contents=[prompt, pdf_file],
                )
            print(f"  Received response from Gemini.")

            # Basic check for response content
            if not response.candidates or not response.candidates[0].content.parts:
                print("  Warning: Gemini response seems empty or malformed.")
                # Check for safety ratings or finish reasons if available
                try:
                    finish_reason = response.candidates[0].finish_reason
                    safety_ratings = response.candidates[0].safety_ratings
                    print(f"  Finish Reason: {finish_reason}")
                    print(f"  Safety Ratings: {safety_ratings}")
                    if response.prompt_feedback:
                        print(f"  Prompt Feedback: {response.prompt_feedback}")
                except (AttributeError, IndexError):
                    pass # Ignore if these attributes don't exist
                return {"error": "Empty or malformed response from API", "raw_response": str(response)}

            extracted_text = response.text
            #print(f"Gemini returned {response.text}")

            # Attempt to parse the JSON response
            try:
                # Clean potential markdown formatting around the JSON
                if extracted_text.strip().startswith("```json"):
                    extracted_text = extracted_text.strip()[7:-3].strip()
                elif extracted_text.strip().startswith("```"):
                    extracted_text = extracted_text.strip()[3:-3].strip()

                result = json.loads(extracted_text)
                for entry in result:
                    results[entity] = result[entry]

            except json.JSONDecodeError:
                print(f"  Warning: Failed to parse JSON response from Gemini.")
                # Log the raw response for debugging
                print(f"  Raw response snippet:\n---\n{textwrap.shorten(extracted_text, width=200, placeholder='...')}\n---")
                #return {"error": "Failed to parse JSON response", "raw_response": extracted_text}
            except Exception as e:
                print(f"  Error processing Gemini response content: {e}")
                #return {"error": f"Error processing response content: {e}", "raw_response": extracted_text}

        except Exception as e:
            # Catch potential API errors (e.g., permission denied on GCS URI, rate limits)
            print(f"  ERROR calling Gemini API for {gcs_uri}: {e}")
            # Check if the error message indicates a permission issue
            if "permission denied" in str(e).lower() or "could not access" in str(e).lower():
                print("  This might be a GCS permission issue. See the note in Cell 3 about granting the 'Storage Object Viewer' role.")
                #return {"error": f"Gemini API call failed: {e}"}
      # Validate that all requested keys are present

    validated_result = {}
    for entity in entities:
        validated_result[entity] = results.get(entity, "n/a") # Default to "Not Found" if key is missing
    return validated_result

In [ ]:
# @title ## KG Integration

from google.colab import files
import pandas as pd

# Upload CSV file
uploaded = files.upload()

# Load the CSV (assumes only one file uploaded)
for filename in uploaded:
    df = pd.read_csv('SemMed.csv')
    print(f"Loaded {filename}")

kg_df = pd.read_csv("SemMed.csv")


Saving SemMed.csv to SemMed.csv
Loaded SemMed.csv


In [ ]:
def get_kg_context(entity, kg_df):
    facts = []
    for _, row in kg_df.iterrows():
        if entity.lower() in row["Node1"].lower() or entity.lower() in row["Node2"].lower():
            fact = f"{row['Node1']} {row['Relationship']} {row['Node2']} (PMID: {row['PMID']})"
            facts.append(fact)
    return "\n".join(facts[:5]) if facts else "None found."



In [ ]:
# @title ## MEREDITH Extractor (S1)

all_results = []
limit = 105

# Initialize the GenAI Client
MODEL_NAME = "gemini-2.0-flash-001"
client = genai.Client(vertexai=True, project=GCP_PROJECT_ID, location=LOCATION)
print(f"Using Gemini model: {MODEL_NAME}")

# Check prerequisites
if not client:
    print("ERROR: GenAI Client not initialized. Please fix configuration.")
elif not storage_client:
    print("ERROR: GCS client not initialized. Please fix configuration.")
else:
    # 1. List PDF files in GCS
    pdf_uris = list_pdfs_in_gcs(storage_client, GCS_BUCKET_NAME, GCS_PREFIX)

if not pdf_uris:
        print("\nNo PDF files found or error listing files. Ensure configuration is correct and files exist.")
else:
        print(f"\nStarting entity extraction for {len(pdf_uris)} PDF files using URIs...")

        # 2. Loop through each PDF URI
        for i, pdf_uri in enumerate(pdf_uris):
          if i > limit:
            break
          print(f"\n--- Processing file {i+1}/{len(pdf_uris)}: {pdf_uri} ---")
          file_result = {"pdf_uri": pdf_uri}

          # 3. Extract Entities using Gemini directly with the URI
          entities_data = extract_entities_with_gemini_uri(pdf_uri, ENTITIES_TO_EXTRACT)
          file_result.update(entities_data)



          if "error" in entities_data:
              print(f"Extraction failed for {pdf_uri}: {entities_data['error']}")
          else:
                print(f"Extraction successful for {pdf_uri}.")

          all_results.append(file_result)

          # Optional: Add a small delay to avoid hitting potential rate limits
          # time.sleep(1) # Sleep for 1 second between requests

        print("\n--- Processing Complete ---")


print(f"\n--- Extracted Information ({len(all_results)} files processed) ---")

# Check if any results were generated
if not all_results:
    print("No results were generated. Please check previous cell outputs for errors.")
else:
    # Iterate and display results for each file
    for result in all_results:
        display(Markdown(f"**File:** `{result.get('pdf_uri', 'N/A')}`")) # Get URI safely
        if "error" in result:
            # Display error message clearly
            display(Markdown(f"  - **Status:** <font color='red'>Error - {result['error']}</font>"))
            # Optionally display raw response snippet if available and useful
            if "raw_response" in result:
                 display(Markdown(f"  - **Raw Response Snippet:**"))
                 display(Markdown(f"```\n{textwrap.shorten(result['raw_response'], width=150, placeholder='...')}\n```"))
        else:
            # Display success and extracted entities
            display(Markdown(f"  - **Status:** <font color='green'>Success</font>"))
            for entity in ENTITIES_TO_EXTRACT:
                value = result.get(entity, "Not Found") # Use .get for safety
                # Format list display (e.g., for authors)
                if isinstance(value, list):
                     display(Markdown(f"  - **{entity}:** {', '.join(map(str, value)) if value else 'None'}")) # Handle non-string list items
                else:
                     display(Markdown(f"  - **{entity}:** {value}"))
        display(Markdown("---")) # Add a separator between file results

Using Gemini model: gemini-2.0-flash-001
Listing PDFs in gs://tlab-extractionbucket3/papers3...
Found 129 PDF files.

Starting entity extraction for 129 PDF files using URIs...

--- Processing file 1/129: gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid and Clear Cell Carcinoma of the Endometrium.pdf ---
  Preparing request for Gemini with URI: gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid and Clear Cell Carcinoma of the Endometrium.pdf
  Calling Gemini API for gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid and Clear Cell Carcinoma of the Endometrium.pdf...
  Processing entity: % of participants with clinicopathological features (Stage I or II)
  Received response from Gemini.
Extraction successful for gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid an

**File:** `gs://tlab-extractionbucket3/papers3/S023_Frequent Mismatch Repair Protein Deficiency in Mixed Endometrioid and Clear Cell Carcinoma of the Endometrium.pdf`

  - **Status:** <font color='green'>Success</font>

  - **% of participants with clinicopathological features (Stage I or II):** {'value': '68%', 'confidence': ['95', 'The text states that 53% of the tumors were in FIGO stage I and 15% of tumors were in FIGO stage II. Adding these values results in 68%.']}

---

...

---

**File:** `gs://tlab-extractionbucket3/papers3/S72_L1 expression as a predictor of progression and survival in patients with uterine and ovarian carcinomas.pdf`

  - **Status:** <font color='green'>Success</font>

  - **% of participants with clinicopathological features (Stage I or II):** {'value': 'n/a', 'confidence': ['0', 'This information is not present in this research paper.']}

---

In [ ]:
# @title ## User Search Integration

def search_extracted_papers(keyword_query, results):
    """Search the extracted entity results for user-specified keyword query.

    Args:
        keyword_query (str): User input in format "Entity: value"
        results (list): The list of extraction result dictionaries (e.g., all_results)

    Returns:
        list: List of matching PDF URIs and their matched entity value.
    """
    import re

    match = re.match(r"^\s*(.+?)\s*:\s*(.+?)\s*$", keyword_query)
    if not match:
        print("Invalid format. Use: Entity: value")
        return []

    entity, value = match.groups()
    entity = entity.strip()
    value = value.strip().lower()

    if not entity or not value:
        print("Invalid input. Provide both an entity and a value.")
        return []

    matches = []
    for result in results:
        entity_data = result.get(entity)
        if not entity_data or isinstance(entity_data, str) and entity_data == "n/a":
            continue

        # entity_data can be a dictionary with 'value' and 'confidence'
        extracted_value = entity_data.get("value") if isinstance(entity_data, dict) else entity_data
        if isinstance(extracted_value, list):
            extracted_value_str = ", ".join(map(str, extracted_value)).lower()
        else:
            extracted_value_str = str(extracted_value).lower()

        if value in extracted_value_str:
            matches.append({
                "pdf_uri": result.get("pdf_uri"),
                "matched_value": extracted_value
            })

    if not matches:
        print("No matches found.")
    else:
        print(f"Found {len(matches)} match(es):")
        for m in matches:
            print(f"- {m['pdf_uri']} → Matched: {m['matched_value']}")

    return matches


In [ ]:
# Example 1: Search for papers that analyzed POLE as a biomarker
search_extracted_papers("Analysed Biomarkers: POLE", all_results)

# Example 2: Search for studies with 192 patients
search_extracted_papers("N of total cohort: 192", all_results)

# Example 3: Search for endometrial studies
search_extracted_papers("Histology: Ovarian", all_results)


No matches found.
No matches found.
Found 3 match(es):
- gs://tlab-extractionbucket/papers/S001_Abundance of mitochondrial superoxide dismutase is a negative predictive biomarker for endometrioisis-associated ovarian cancers.pdf → Matched: Ovarian
- gs://tlab-extractionbucket/papers/S002_ADP-ribosylation-factor-like-4C-EAOC.pdf → Matched: Ovarian
- gs://tlab-extractionbucket/papers/S003_Nuclear β-catenin and CDX2 expression in ovarian endometrioid carcinoma identify patients with favourable outcome.pdf → Matched: Ovarian


[{'pdf_uri': 'gs://tlab-extractionbucket/papers/S001_Abundance of mitochondrial superoxide dismutase is a negative predictive biomarker for endometrioisis-associated ovarian cancers.pdf',
  'matched_value': 'Ovarian'},
 {'pdf_uri': 'gs://tlab-extractionbucket/papers/S002_ADP-ribosylation-factor-like-4C-EAOC.pdf',
  'matched_value': 'Ovarian'},
 {'pdf_uri': 'gs://tlab-extractionbucket/papers/S003_Nuclear β-catenin and CDX2 expression in\xa0ovarian\xa0endometrioid\xa0carcinoma\xa0identify patients with favourable outcome.pdf',
  'matched_value': 'Ovarian'}]

In [ ]:
pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.8 MB/s eta 0:00:00


In [ ]:
# ENTITIES_TO_EXTRACT = {
# "Title of Paper": "Extract the title of the paper, e.g., POLE exonuclease domain mutation predicts long progression-free survival in grade 3 endometrioid carcinoma of the endometrium",
# "Analysed Biomarkers":"Extract which biomarkers were analysed in the study, e.g., POLE, p53, MMR, L1CAM, BRCA1. Only extract the top 3 most significant biomarkers, often mentioned in the methods and/or abstract and/or conclusion and analysing the surrounding text if the biomarker is found in the abstract to make sure it is not mentioned as contextual information, and is actually being analyzed in the paper.",
#  "Biomarker combinations":"Based on the biomarkers studied in this paper, what combinations of biomarkers were analyzed together in the same context (e.g., same figure, table, or statistical model)?",
# "Survival analysis": (
#         "Check for univariable survival analyses done for each biomarker. "
#         "For each, specify what survival variable was tested (e.g., OS, PFS) and report the p-value. "
#         "Then, check whether a multivariable survival analysis was performed. If yes, state which biomarkers were included and their p-values. "
#         "Summarize key survival findings clearly."
#     ),
# "Confounding factors": "Were any confounding factors (e.g., age, stage, histotype) considered in the survival analysis involving biomarker combinations? Return 'Yes' or 'No'. Also, how many different combinations of biomarkers were analyzed together?",
# "Primary biomarker": "Identify the primary biomarker studied in this paper (i.e., the one most emphasized in title, abstract, or results). List additional biomarkers also studied and the total number of mentions or contexts for each (e.g., subgroup analysis, comparison).",
# "Key Findings": "Return the key overeall findings relating to the biomarkers from the paper. They are most likely mentioned in the Results section/"
# }

In [ ]:
# @title ## MEREDITH Extractor (S2 Test)

all_results = []
limit = 11

# Check prerequisites
if not client:
    print("ERROR: GenAI Client not initialized. Please fix configuration.")
elif not storage_client:
    print("ERROR: GCS client not initialized. Please fix configuration.")
else:
    # 1. List PDF files in GCS
  pdf_files_to_process = [
        "S008_S100A1 Expression in Ovarian and Endometrial Endometrioid Carcinomas Is a Prognostic Indicator of Relapse-Free Survival.pdf",
        "S011_POLE exonuclease domain mutation predicts long progression-free survival in grade 3 endometrioid carcinoma of the endometrium.pdf",
        "S015_Molecular classification defines outcomes and opportunities in young women with endometrial carcinoma.pdf",
        "S018_Molecular Classification of Grade 3 Endometrioid Endometrial Cancers Identifies Distinct Prognostic Subgroups.pdf",
        "S019_L1CAM further stratifies endometrial carcinoma patients with no specific molecular risk profile.pdf",
        "S021_Confirmation of ProMisE- A simple, genomics-based clinical classifier for endometrial cancer.pdf",
        "S022_Evaluation of endometrial carcinoma prognostic immunohistochemistry markers in the context of molecular classification.pdf",
        "S024_Markers of the p53 pathway further refine molecular profiling in high-risk endometrial cancer- A TransPORTEC initiative.pdf",
        "S025_Endometrial Carcinomas with POLE Exonuclease Domain Mutations Have a Favorable Prognosis.pdf",
        "S30_Integrated genomic characterization of endometrial carcinoma.pdf",
    ]

  pdf_uris = list_pdfs_in_gcs(storage_client, GCS_BUCKET_NAME, GCS_PREFIX)

  pdf_uris = [
        uri for uri in pdf_uris
        if any(uri.endswith(filename) for filename in pdf_files_to_process)
    ]

if not pdf_uris:
        print("\nNo PDF files found or error listing files. Ensure configuration is correct and files exist.")
else:
        print(f"\nStarting entity extraction for {len(pdf_uris)} PDF files using URIs...")

        # 2. Loop through each PDF URI
        for i, pdf_uri in enumerate(pdf_uris):
          if i > limit:
            break
          print(f"\n--- Processing file {i+1}/{len(pdf_uris)}: {pdf_uri} ---")
          file_result = {"pdf_uri": pdf_uri}

          # 3. Extract Entities using Gemini directly with the URI
          entities_data = extract_entities_with_gemini_uri(pdf_uri, ENTITIES_TO_EXTRACT)
          file_result.update(entities_data)



          if "error" in entities_data:
              print(f"Extraction failed for {pdf_uri}: {entities_data['error']}")
          else:
                print(f"Extraction successful for {pdf_uri}.")

          all_results.append(file_result)

          # Optional: Add a small delay to avoid hitting potential rate limits
          # time.sleep(1) # Sleep for 1 second between requests

        print("\n--- Processing Complete ---")


print(f"\n--- Extracted Information ({len(all_results)} files processed) ---")

# Check if any results were generated
if not all_results:
    print("No results were generated. Please check previous cell outputs for errors.")
else:
    # Iterate and display results for each file
    for result in all_results:
        display(Markdown(f"**File:** `{result.get('pdf_uri', 'N/A')}`")) # Get URI safely
        if "error" in result:
            # Display error message clearly
            display(Markdown(f"  - **Status:** <font color='red'>Error - {result['error']}</font>"))
            # Optionally display raw response snippet if available and useful
            if "raw_response" in result:
                 display(Markdown(f"  - **Raw Response Snippet:**"))
                 display(Markdown(f"```\n{textwrap.shorten(result['raw_response'], width=150, placeholder='...')}\n```"))
        else:
            # Display success and extracted entities
            display(Markdown(f"  - **Status:** <font color='green'>Success</font>"))
            for entity in ENTITIES_TO_EXTRACT:
                raw_entity = result.get(entity, "Not Found")

                if isinstance(raw_entity, dict):
                    value = raw_entity.get("value", "Not Found")
                    confidence = raw_entity.get("confidence", None)
                else:
                    value = raw_entity
                    confidence = None


                    # Format list display (e.g., for authors)
                if isinstance(value, list):
                    value_str = ', '.join(map(str, value)) if value else 'None'
                else:
                    value_str = str(value)

                confidence_str = f" _(Confidence: {confidence}%)_" if confidence is not None else ""

                display(Markdown(f"  - **{entity}:** {value_str}{confidence_str}"))

        display(Markdown("---")) # Add a separator between file results



Listing PDFs in gs://tlab-extractionbucket/papers...
Found 25 PDF files.

Starting entity extraction for 10 PDF files using URIs...

--- Processing file 1/10: gs://tlab-extractionbucket/papers/S008_S100A1 Expression in Ovarian and Endometrial Endometrioid Carcinomas Is a Prognostic Indicator of Relapse-Free Survival.pdf ---
  Preparing request for Gemini with URI: gs://tlab-extractionbucket/papers/S008_S100A1 Expression in Ovarian and Endometrial Endometrioid Carcinomas Is a Prognostic Indicator of Relapse-Free Survival.pdf
  Calling Gemini API for gs://tlab-extractionbucket/papers/S008_S100A1 Expression in Ovarian and Endometrial Endometrioid Carcinomas Is a Prognostic Indicator of Relapse-Free Survival.pdf...
  Processing entity: Title of Paper
  Received response from Gemini.
  Processing entity: Analysed Biomarkers
  Received response from Gemini.
  Processing entity: Biomarker combinations
  Received response from Gemini.
  Processing entity: Survival analysis
  Received response 

**File:** `gs://tlab-extractionbucket/papers/S008_S100A1 Expression in Ovarian and Endometrial Endometrioid Carcinomas Is a Prognostic Indicator of Relapse-Free Survival.pdf`

  - **Status:** <font color='green'>Success</font>

  - **Title of Paper:** S100A1 Expression in Ovarian and Endometrial Endometrioid Carcinomas Is a Prognostic Indicator of Relapse-Free Survival _(Confidence: ['100', 'The title is clearly visible at the beginning of the document.']%)_

  - **Analysed Biomarkers:** S100A1, CD3, HDAC1 _(Confidence: ['85', 'S100A1 is listed in the title and abstract, and CD3 and HDAC1 were found by using the same TMAs as the study. ']%)_

...

  - **Survival analysis:** The research paper mentions progression-free survival (PFS) analyses in relation to copy number clusters and POLE mutations but does not present the p-values or the biomarkers used in multivariable analysis. For example, the paper states, "Copy number clusters 2 and 3 consisted mainly of endometrioid tumors, distinguished by more frequent 1q amplification in cluster 3 than cluster 2 (100% of cluster 3 tumors vs. 33% of cluster 2 tumors) and worse progression-free survival (PFS, P=0.003, log-rank vs. clusters 1 and 2; Fig. 1b)." It also states, "The ultra-mutated group consisted of 17 (7%) tumors exemplified by an increased C→A transversion frequency, all with mutations in the exonuclease domain of POLE, and an improved progression-free survival (Fig. 2a and 2c)." _(Confidence: ['80', 'The text mentions survival analysis being performed and indicates log-rank p-values, however, further details regarding the specific details of univariable and multivariable analysis are not given.']%)_

  - **Confounding factors:** No. The paper does not explicitly state that confounding factors were considered in the survival analysis. However, it does state that clinical and pathologic characteristics of the samples were representative of the population of interest (Supplementary Table 1.1). Also, four different combinations of biomarkers, as identified in Figure 2, were analyzed. _(Confidence: ['75', 'While the paper does not explicitly mention correcting for confounding factors in the survival analysis, it thoroughly integrates genomic, transcriptomic, and proteomic data to characterize endometrial carcinoma subtypes. It may be considered that correcting for these characteristics when creating the four different groups would then be considered correcting for confounding factors.']%)_

  - **Primary biomarker:** POLE _(Confidence: ['90', 'POLE mutations were emphasized throughout the paper, including classification into POLE ultramutated tumors.']%)_

  - **Key Findings:** Uterine serous tumors and ~25% of high-grade endometrioid tumors have extensive copy number alterations, few DNA methylation changes, low ER/PR levels, and frequent TP53 mutations. Most endometrioid tumors have few copy number alterations or TP53 mutations but frequent mutations in PTEN, CTNNB1, PIK3CA, ARIDIA, KRAS and novel mutations in the SWI/SNF gene ARID5B. A subset of endometrioid tumors identified had a dramatically increased transversion mutation frequency, and newly identified hotspot mutations in POLE. Results classified endometrial cancers into four categories: POLE ultramutated, microsatellite instability hypermutated, copy number low, and copy number high. Uterine serous carcinomas share genomic features with ovarian serous and basal-like breast carcinomas. The genomic features of endometrial carcinomas permit a reclassification that may impact post-surgical adjuvant treatment for women with aggressive tumors. _(Confidence: ['100', 'This text is from the summary section of the paper so is considered a key finding']%)_

---

In [ ]:
# @title ## MEREDITH Extractor (S2)

from IPython.display import display, Markdown
import time
from io import BytesIO
from PyPDF2 import PdfReader
import re
import textwrap
import google.generativeai as genai

genai.configure(api_key='')
model = genai.GenerativeModel("gemini-2.0-flash-001")

# Custom biomarker follow-up prompts
BIOMARKER_ENTITIES_TO_EXTRACT = {
      "Individual gene markers or biomarker signatures (univariate)": "Extract and list the individual gene markers tested within the study either by biomarker name or official gene name.",
     "Biomarker signatures (multivariable)": "Identify and extract the multivariable analysis conducted with each biomarker. For example, whether positive vs negative expression was tested, mutant vs wildtype, mutational status within subgroup, or deficient (d) vs proficient.",
     "Within the context of ProMisE": "Given the biomarker, determine if it is used in the ProMisE molecular classification of endometrial cancer. ProMisE uses three biomarker categories in sequence:\n\nMismatch Repair (MMR) proteins: MLH1, MSH2, MSH6, PMS2\n\nPOLE exonuclease domain mutations (EDM)\n\np53 immunohistochemistry (IHC)\n\nReturn 'Yes' if the biomarker belongs to one of these three categories, otherwise return 'No'.",
   # "Biomarker combinations": "Based on the biomarkers studied in this paper, what combinations of biomarkers were analyzed together in the same context (e.g., same figure, table, or statistical model)?",
   # "Survival analysis": "For each combination of biomarkers analyzed in this study, was a survival analysis (e.g., overall survival, progression-free survival) performed? If yes, summarize the result briefly for each combination.",
   # "Confounding factors": "Were any confounding factors (e.g., age, stage, histotype) considered in the survival analysis involving biomarker combinations? Return 'Yes' or 'No'. Also, how many different combinations of biomarkers were analyzed together?",
   # "Primary biomarker": "Identify the primary biomarker studied in this paper (i.e., the one most emphasized in title, abstract, or results). List additional biomarkers also studied and the total number of mentions or contexts for each (e.g., subgroup analysis, comparison).",
}

all_results = []
limit = 50

if not client:
    print("ERROR: GenAI Client not initialized. Please fix configuration.")
elif not storage_client:
    print("ERROR: GCS client not initialized. Please fix configuration.")
else:
    pdf_uris = list_pdfs_in_gcs(storage_client, GCS_BUCKET_NAME, GCS_PREFIX)

    pdf_files_to_process = [
        "S008_S100A1 Expression in Ovarian and Endometrial Endometrioid Carcinomas Is a Prognostic Indicator of Relapse-Free Survival.pdf",
        # Add other filenames if needed
    ]

    pdf_uris = [
        uri for uri in pdf_uris
        if any(uri.endswith(filename) for filename in pdf_files_to_process)
    ]

    if not pdf_uris:
        print("\nNo PDF files found or error listing files. Ensure configuration is correct and files exist.")
    else:
        print(f"\nStarting extraction for {len(pdf_uris)} PDF files using URIs...")

        for i, pdf_uri in enumerate(pdf_uris):
            if i > limit:
                break

            print(f"\n--- Processing file {i+1}/{len(pdf_uris)}: {pdf_uri} ---")
            file_result = {"pdf_uri": pdf_uri}

            # Step 1: Extract core entities
            entities_data = extract_entities_with_gemini_uri(pdf_uri, ENTITIES_TO_EXTRACT)
            file_result.update(entities_data)

            if "error" in entities_data:
                print(f"Extraction failed for {pdf_uri}: {entities_data['error']}")
            else:
                print(f"Extraction successful for {pdf_uri}.")

                # Step 2: Biomarker follow-up questions
                raw_biomarkers = entities_data.get("Analysed Biomarkers", "")

                # Improved parsing of biomarkers to avoid splitting gene names
                if isinstance(raw_biomarkers, str):
                    if "," in raw_biomarkers:
                        biomarkers = [b.strip() for b in raw_biomarkers.split(",") if b.strip()]
                    else:
                        biomarkers = [raw_biomarkers.strip()]
                elif isinstance(raw_biomarkers, list):
                    biomarkers = [b.strip() for b in raw_biomarkers if b.strip()]
                else:
                    biomarkers = []

                biomarkers = list(dict.fromkeys(biomarkers))  # Deduplicate preserving order

                # 🔸 NEW: Get full paper text from GCS and extract text
                try:
                    def extract_text_from_gcs(storage_client, gcs_uri):
                        from PyPDF2 import PdfReader
                        from io import BytesIO
                        bucket_name, path = gcs_uri.replace("gs://", "").split("/", 1)
                        bucket = storage_client.bucket(bucket_name)
                        blob = bucket.blob(path)
                        pdf_bytes = blob.download_as_bytes()
                        reader = PdfReader(BytesIO(pdf_bytes))
                        return "".join([page.extract_text() or "" for page in reader.pages])

                    pdf_text = extract_text_from_gcs(storage_client, pdf_uri)
                except Exception as e:
                    pdf_text = ""
                    print(f"[ERROR] Failed to extract PDF text for {pdf_uri}: {e}")

                biomarker_results = []
                paper_title = entities_data.get("Title of Paper", "this study")

                for biomarker in biomarkers:
                    for label, prompt_template in BIOMARKER_ENTITIES_TO_EXTRACT.items():
                        formatted_prompt = (
                        f"In the context of the following study text, "
                        f"regarding the biomarker '{biomarker}':\n\n{prompt_template}\n\n"
                        f"Please respond in 1–2 sentences without any explanation. "
                        f"Also provide a confidence score (0–100) on a new line labeled 'Confidence Score:' and a short, one sentence rationale for the given confidence score.\n\n"
                        f"{pdf_text[:8000]}"
                    )

                        try:
                            response = model.generate_content(formatted_prompt)
                            time.sleep(1.5)

                            response_text = getattr(response, "text", None)
                            if not response_text:
                                raise ValueError("Gemini response has no text.")

                            match = re.search(r"confidence score[^0-9]*([0-9]{1,3})", response_text, re.IGNORECASE)
                            confidence = int(match.group(1)) if match else None

                            biomarker_results.append({
                                "biomarker": biomarker,
                                "label": label,
                                "prompt": formatted_prompt,
                                "response": response_text.strip(),
                                "confidence": confidence
                            })

                        except Exception as e:
                            biomarker_results.append({
                                "biomarker": biomarker,
                                "label": label,
                                "prompt": formatted_prompt,
                                "response": f"ERROR: {str(e)}"
                            })

                file_result["biomarker_followups"] = biomarker_results

            all_results.append(file_result)


print(f"\n--- Extracted Information ({len(all_results)} files processed) ---")

if not all_results:
    print("No results were generated. Please check previous cell outputs for errors.")
else:
    for result in all_results:
        title = result.get("Title of Paper", "N/A")
        raw_biomarkers = result.get("Analysed Biomarkers", "")

        # Format biomarkers nicely for display (reuse parsing logic)
        if isinstance(raw_biomarkers, str):
            if "," in raw_biomarkers:
                biomarkers = [b.strip() for b in raw_biomarkers.split(",") if b.strip()]
            else:
                biomarkers = [raw_biomarkers.strip()]
        elif isinstance(raw_biomarkers, list):
            biomarkers = [b.strip() for b in raw_biomarkers if b.strip()]
        else:
            biomarkers = []

        biomarkers = list(dict.fromkeys(biomarkers))

        followups = result.get("biomarker_followups", [])

        display(Markdown(f"### **Title of paper:**\n{title}"))
        display(Markdown(f"**Analysed biomarkers:** {', '.join(biomarkers) if biomarkers else 'None'}"))

        # Group responses by biomarker
        biomarker_grouped = {}
        for entry in followups:
            biomarker = entry["biomarker"]
            if biomarker not in biomarker_grouped:
                biomarker_grouped[biomarker] = {}
            biomarker_grouped[biomarker][entry["label"]] = entry["response"]

        for biomarker, responses in biomarker_grouped.items():
            display(Markdown(f"\n---\n### **{biomarker}:**"))
            for label in BIOMARKER_ENTITIES_TO_EXTRACT:
                response_text = responses.get(label, "Not found.")
                display(Markdown(f"**{label}:**\n{response_text}"))

        display(Markdown("---\n"))


ModuleNotFoundError: No module named 'PyPDF2'

In [ ]:
import csv
import pandas as pd

# Replace this with your actual list of dictionaries containing extracted data
# For example: extracted_data = [{'Title of Paper': '...', 'Author list': '...', ...}, ...]
extracted_data = all_results
csv_file_path = 'extracted_results.csv'

# Get all unique fieldnames from your data
fieldnames = list({key for d in extracted_data for key in d.keys()})

with open(csv_file_path, mode='w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for row in extracted_data:
        writer.writerow(row)

print(f"\nExported extracted results to {csv_file_path}")



Exported extracted results to extracted_results.csv


In [ ]:
#S1 data visualization
import pandas as pd

# Convert data to DataFrame and save to CSV
df = pd.DataFrame(extracted_data)
df.to_csv('extracted_results.csv', index=False)

# Set row numbers (index) to start at 1
df.index = range(1, len(df) + 1)

# Display all rows
pd.set_option('display.max_rows', None)
df


,pdf_uri,% of participants with clinicopathological features (Stage I or II)
1,gs://tlab-extractionbucket3/papers3/S023_Frequ...,"{'value': '68%', 'confidence': ['90', 'The tex..."
2,gs://tlab-extractionbucket3/papers3/S024_Marke...,"{'value': 'n/a', 'confidence': ['0', 'The pape..."
3,gs://tlab-extractionbucket3/papers3/S025_Endom...,"{'value': '95%', 'confidence': ['95', 'The tex..."
4,gs://tlab-extractionbucket3/papers3/S026_Assoc...,"{'value': 'n/a', 'confidence': ['0', 'The pape..."
5,gs://tlab-extractionbucket3/papers3/S027_Sex S...,"{'value': '13%', 'confidence': ['100', 'This v..."
6,gs://tlab-extractionbucket3/papers3/S028_PIK3C...,"{'value': 'n/a', 'confidence': ['0', 'The pape..."
7,gs://tlab-extractionbucket3/papers3/S029_MMR d...,{'value': '46 Stage I + 13 Stage II / 102 tota...
8,gs://tlab-extractionbucket3/papers3/S30_Integr...,"{'value': 'n/a', 'confidence': ['0', 'I did no..."
9,gs://tlab-extractionbucket3/papers3/S31_ARID1A...,"{'value': 'n/a', 'confidence': ['0', 'The pape..."
10,gs://tlab-extractionbucket3/papers3/S32_Necros...,"{'value': '81%', 'confidence': ['100', '58% + ..."


In [ ]:
#S2 Data visualization
import pandas as pd

# Prepare structured flat list for DataFrame
flattened_rows = []

for result in all_results:
    title = result.get("Title of Paper", "N/A")
    all_biomarkers = result.get("Analysed Biomarkers", [])
    followups = result.get("biomarker_followups", [])

    if isinstance(all_biomarkers, list):
        analysed_str = ", ".join(all_biomarkers)
    else:
        analysed_str = str(all_biomarkers)

    # Group followups by biomarker
    grouped = {}
    for entry in followups:
        biomarker = entry["biomarker"]
        if biomarker not in grouped:
            grouped[biomarker] = {}
        grouped[biomarker][entry["label"]] = entry["response"]

    for biomarker, labels in grouped.items():
        row = {
            "Title of Paper": title,
            "Analysed Biomarkers": analysed_str,
            "Biomarker": biomarker,
            "Univariate": labels.get("Individual gene markers or biomarker signatures (univariate)", "Not found."),
            "Multivariate": labels.get("Biomarker signatures (multivariable)", "Not found."),
            "Within the context of ProMisE": labels.get("Within the context of ProMisE", "Not found.")
        }
        flattened_rows.append(row)

# Create DataFrame
df = pd.DataFrame(flattened_rows)

# Optional: reset index starting at 1
df.index = range(1, len(df) + 1)

# Optional: display all rows
pd.set_option('display.max_rows', None)

# Save to CSV
df.to_csv("extracted_results.csv", index=False)

# Display the DataFrame
df


,Title of Paper,Analysed Biomarkers,Biomarker,Univariate,Multivariate,Within the context of ProMisE
1,S100A1 Expression in Ovarian and Endometrial E...,S100A1,S100A1,"Based on the provided text, the gene markers t...","Based on the text provided, here's the multiva...",No
2,POLE exonuclease domain mutation predicts long...,"POLE, MSH6, PIK3CA",POLE,"Based on the provided text, the individual gen...","Based on the provided text, here's the multiva...",Yes
3,POLE exonuclease domain mutation predicts long...,"POLE, MSH6, PIK3CA",MSH6,"Based on the provided text, the gene markers t...","Based on the provided text, here's the multiva...",Yes
4,POLE exonuclease domain mutation predicts long...,"POLE, MSH6, PIK3CA",PIK3CA,"Based on the provided text, the following gene...","Based on the provided text, here's the multiva...",No
5,Molecular classification defines outcomes and ...,"MMR, POLE, p53",MMR,"Based on the provided text, the MMR biomarker ...","Based on the provided text, here's the informa...",Yes
6,Molecular classification defines outcomes and ...,"MMR, POLE, p53",POLE,"Based on the provided text, the individual gen...",ERROR: 429 POST https://generativelanguage.goo...,ERROR: 429 POST https://generativelanguage.goo...
7,Molecular classification defines outcomes and ...,"MMR, POLE, p53",p53,"Based on the provided text, the following gene...",ERROR: 429 POST https://generativelanguage.goo...,ERROR: 429 POST https://generativelanguage.goo...
